# Manually pipelining tasks: Neuroimaging example

## Summary

In this *Notebook* you'll be able to execute two subsequent operations on anatomical T<sub>1</sub>-weighted MRI images of adult human heads.

The first typical operation is *brain extraction*, which involves identifying brain tissue and cerebrospinal fluid (CSF) in the image, and discard non-brain tissue (e.g. skull, neck, skin and fat, etc.)

Only once the brain has been *extracted*, we can use machine learning to identify three major types of tissue: gray matter (GM), white matter and CSF.

## Setting up

First, we import `Path` from *Python*'s standard library, and the `NiiVue` class from the *ipyniivue* library for visualization of 3D brain images.

Second, we define some paths (`DATA_PATH` and `OUTPUTS_PATH`) where we will be finding and storing data (respectively).

In [ ]:
from pathlib import Path
from ipyniivue import NiiVue

In [ ]:
DATA_PATH = Path.home() / "data" / "ds000005"
OUTPUTS_PATH = Path.home() / "outputs"

## Visualizing a T1w image of the human brain

The interactive viewer allows for navigation of three plane cuts through the volume, as well as a 3D rendering.

In [ ]:
nv = NiiVue()
nv.load_volumes([{"path": DATA_PATH / "sub-01/anat/sub-01_T1w.nii.gz"}])
nv

## Running a brain extraction program

We will extract the brain using `mri_synthstrip`, from [FreeSurfer](https://surfer.nmr.mgh.harvard.edu/). It is a neural network trained on synthesised images, which is why it copes with almost any contrast without being told what it is looking at.

We can execute shell commands from within the Jupyter Notebook (that is, code that *is not* Python) by preceding the line with an exclamation mark `!`, so `!mri_synthstrip` tells this notebook to run that on a terminal shell.

As an extra exercise, create a new cell and read the tool's own help:

```Bash
!mri_synthstrip --help
```

> **If the next cell is killed without an error message**, your Docker has less memory than the network needs — it wants about 6 GB. Either raise it (Docker Desktop → Settings → Resources → Memory) or use `simple_strip`, which does the same job with thresholding and morphology in about 300 MB:
>
> ```Bash
> !simple_strip -i $HOME/data/ds000005/sub-01/anat/sub-01_T1w.nii.gz -o $HOME/outputs/sub-01_desc-brain_T1w.nii.gz
> ```
>
> It is visibly worse. Comparing the two masks is a better exercise than trusting either.

In [ ]:
!mri_synthstrip -i $HOME/data/ds000005/sub-01/anat/sub-01_T1w.nii.gz -o $HOME/outputs/sub-01_desc-brain_T1w.nii.gz

## Visualizing the output

We can now look at the result of executing `mri_synthstrip` on a head image.

In [ ]:
nv = NiiVue()
nv.load_volumes([{"path": OUTPUTS_PATH / "sub-01_desc-brain_T1w.nii.gz"}])
nv

## Identifying brain tissues

The second step labels each remaining voxel as CSF, grey matter or white matter, with `tissue_segment`.

It is worth knowing what it does, because it is about forty lines and you can read all of them. It fits a **mixture of three Gaussians** to the intensities of the voxels the brain extraction left behind, and then assigns each voxel to whichever Gaussian most likely produced it. The model knows nothing about anatomy. The anatomy enters on exactly one line, where the three fitted means are sorted: in a T1-weighted image CSF is dark, white matter is bright, and grey matter sits between them, so the darkest component *is* the CSF.

That is also its limit. Three Gaussians on intensity alone cannot see that tissue comes in connected sheets, so the result is noisier than a real segmentation tool's. Those tools carry a spatial prior; this one does not.

The third task, `tissue_volumes`, does no science at all: it counts voxels and multiplies by how big a voxel is. Most of the tasks in a real pipeline are that kind of task.

In [ ]:
!tissue_segment $HOME/outputs/sub-01_desc-brain_T1w.nii.gz $HOME/outputs/sub-01_seg-tissues_dseg.nii.gz

In [ ]:
!tissue_volumes $HOME/outputs/sub-01_seg-tissues_dseg.nii.gz $HOME/outputs/sub-01_seg-tissues_volumes.tsv

## Visualizing the separated brain tissues

Now we can load the output after conversion and we can see how the segmentation performed.

In [ ]:
nv = NiiVue()
nv.load_volumes([{"path": OUTPUTS_PATH / "sub-01_seg-tissues_dseg.nii.gz", "colormap": "nih"}])
nv

# Lines for future progress:

- (Easy) Benchmark the latency of each task using `%%time`
- (Easy) Open the brain tissue segmentation with nibabel and check the volumes against what `tissue_volumes` reported
- (Easy) Play with the colormap of the different visualizations
- (Easy) Run the segmentation again with `--vb`, which swaps EM for a variational Bayesian mixture, and explain why the grey-to-white ratio changes so much
- (Easy) Download more data. The dataset is served over plain HTTPS:
  ```Bash
  # Subject 02's anatomical
  !curl -fsSL -o $HOME/data/sub-02_T1w.nii.gz \
      https://s3.amazonaws.com/openneuro.org/ds000005/sub-02/anat/sub-02_T1w.nii.gz
  ```
- (Easy) Run this workflow manually on subject 2's anatomical image
- (Medium) Compare `mri_synthstrip` and `simple_strip`: write both masks with `-m`, and compute the Dice coefficient between them
- (Medium) Write Python code to calculate the voxel-wise average across time of one of subject 1's functional images
- (Medium) Extract a brain mask from that functional average with `simple_strip`
- (Medium-hard) Rewrite the pipeline using Python's `subprocess`